In [ ]:
# requirements:
# pip install pandas numpy sentence-transformers pyarrow

import numpy as np
import pandas as pd
from typing import List, Dict, Tuple
from sentence_transformers import SentenceTransformer

def rank_locations_for_phrases(
    phrases_df: pd.DataFrame,
    phrase_col: str | None = None,
    emb_npy_path: str = "../embeddings/Review_embeddings/review_embeddings.npy",
    meta_csv_path: str = "../embeddings/Review_embeddings/meta.csv",
    model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
    similarity_threshold: float = 0.35,
    top_n_per_location: int = 10,
    min_reviews_per_location: int = 5,
    k_top_locations: int = 5,
    location_col: str = "location",   # <-- matches your meta.csv
) -> pd.DataFrame:
    """Rank locations for the given phrases using review-level aggregation."""

    # --- helpers ---
    def l2_normalize(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
        norms = np.linalg.norm(X, axis=1, keepdims=True)
        return X / np.maximum(norms, eps)

    def embed_phrases(model: SentenceTransformer, phrases: List[str]) -> np.ndarray:
        vecs = model.encode(
            phrases, batch_size=32, convert_to_numpy=True,
            show_progress_bar=False, normalize_embeddings=False
        ).astype(np.float32)
        return l2_normalize(vecs)

    def load_review_embeddings_and_meta(emb_npy: str, meta_csv: str) -> Tuple[np.ndarray, pd.DataFrame]:
        review_vecs = np.load(emb_npy).astype(np.float32)
        meta = pd.read_csv(meta_csv)
        if len(meta) != len(review_vecs):
            raise ValueError(f"Embeddings rows ({len(review_vecs)}) != meta rows ({len(meta)})")
        # defensively normalize (harmless if already normalized)
        review_vecs = l2_normalize(review_vecs)
        return review_vecs, meta

    def similarity_matrix(phrase_vecs: np.ndarray, review_vecs: np.ndarray) -> np.ndarray:
        return phrase_vecs @ review_vecs.T  # cosine == dot (both L2-normalized)

    def aggregate_by_location(
        sims_row: np.ndarray,
        meta: pd.DataFrame,
        loc_col: str,
        threshold: float,
        top_n: int,
        min_reviews: int,
    ) -> Dict[str, float]:
        mask = sims_row >= threshold
        if not mask.any():
            return {}
        sims_kept = sims_row[mask]
        locs_kept = meta.loc[mask, loc_col].to_numpy()

        order = np.argsort(-sims_kept)  # desc
        sims_sorted = sims_kept[order]
        locs_sorted = locs_kept[order]

        counts: Dict[str, int] = {}
        buckets: Dict[str, List[float]] = {}
        for sim, loc in zip(sims_sorted, locs_sorted):
            c = counts.get(loc, 0)
            if c < top_n:
                buckets.setdefault(str(loc), []).append(float(sim))
                counts[loc] = c + 1

        return {loc: float(np.mean(vals)) for loc, vals in buckets.items() if len(vals) >= min_reviews}

    def normalize_scores_per_phrase(scores: Dict[str, float]) -> Dict[str, float]:
        if not scores:
            return {}
        v = np.array(list(scores.values()), dtype=np.float32)
        vmin, vmax = float(v.min()), float(v.max())
        if vmax <= vmin:
            return {k: 0.0 for k in scores}
        return {k: (scores[k] - vmin) / (vmax - vmin) for k in scores}

    def combine_phrase_scores(per_phrase_norm: List[Dict[str, float]]) -> Dict[str, float]:
        if not per_phrase_norm:
            return {}
        all_locs = set().union(*[d.keys() for d in per_phrase_norm])
        return {loc: float(np.mean([d.get(loc, 0.0) for d in per_phrase_norm])) for loc in all_locs}

    # --- phrases (assume 1 col; allow named or unnamed) ---
    if phrase_col is None:
        if phrases_df.shape[1] != 1:
            raise ValueError("phrases_df must have a single text column or specify phrase_col.")
        phrase_col = phrases_df.columns[0]
    phrases = phrases_df[phrase_col].astype(str).str.strip().tolist()
    if not phrases:
        raise ValueError("No phrases provided.")

    # --- embed phrases ---
    model = SentenceTransformer(model_name)
    phrase_vecs = embed_phrases(model, phrases)

    # --- load artifacts ---
    review_vecs, meta = load_review_embeddings_and_meta(emb_npy_path, meta_csv_path)
    if location_col not in meta.columns:
        raise KeyError(f"location_col '{location_col}' not found. Meta columns: {list(meta.columns)}")

    # --- similarity + per-phrase scoring ---
    S = similarity_matrix(phrase_vecs, review_vecs)
    per_phrase_scores = [
        aggregate_by_location(S[i], meta, location_col, similarity_threshold, top_n_per_location, min_reviews_per_location)
        for i in range(S.shape[0])
    ]

    # --- normalize, combine, build result ---
    per_phrase_norm = [normalize_scores_per_phrase(d) for d in per_phrase_scores]
    combined = combine_phrase_scores(per_phrase_norm)

    rows = []
    for loc, score in combined.items():
        row = {"location": loc, "combined_score": score}
        for idx, d in enumerate(per_phrase_norm, start=1):
            row[f"phrase_{idx}_score"] = d.get(loc, 0.0)
        rows.append(row)

    result_df = pd.DataFrame(rows)
    if result_df.empty:
        cols = ["rank", "location", "combined_score"] + [f"phrase_{i+1}_score" for i in range(len(phrases))]
        return pd.DataFrame(columns=cols)

    result_df = result_df.sort_values("combined_score", ascending=False).reset_index(drop=True)
    result_df.insert(0, "rank", np.arange(1, len(result_df) + 1))
    return result_df.head(k_top_locations)


In [ ]:
phrases_df = pd.DataFrame({
    0: [
        "a lively city where someone could enjoy street food",
        "a place with landmarks where someone could do sightseeing",
        "a scene of a river",
        "a scenic place where someone could do photography",
        "a city, town or resort where someone could go shopping",
    ]
})
ranked_locations = rank_locations_for_phrases(
    phrases_df,
    emb_npy_path="../embeddings/Review_embeddings/review_embeddings.npy",
    meta_csv_path="../embeddings/Review_embeddings/meta.csv",
    similarity_threshold=0.35,
    top_n_per_location=10,
    min_reviews_per_location=5,
    k_top_locations=5,
    location_col="location",
)
print(ranked_locations)


   rank     location  combined_score  phrase_1_score  phrase_2_score  \
0     1        Seoul        0.808158        0.995065        0.743687   
1     2  Quebec City        0.723928        0.435570        0.945180   
2     3       London        0.703497        1.000000        0.653213   
3     4  Jeju Island        0.640966        0.448359        0.893056   
4     5        Cairo        0.639376        0.521473        0.758254   

   phrase_3_score  phrase_4_score  phrase_5_score  
0        0.701232        0.631620        0.969186  
1        1.000000        0.725067        0.513826  
2        0.737227        0.498186        0.628858  
3        0.357538        0.902597        0.603282  
4        0.583601        0.576617        0.756938  


In [16]:
out.head()

,rank,location,combined_score,phrase_1_score,phrase_2_score,phrase_3_score,phrase_4_score,phrase_5_score
0,1,Seoul,0.808158,0.995065,0.743687,0.701232,0.631620,0.969186
1,2,Quebec City,0.723928,0.435570,0.945180,1.000000,0.725067,0.513826
2,3,London,0.703497,1.000000,0.653213,0.737227,0.498186,0.628858
3,4,Jeju Island,0.640966,0.448359,0.893056,0.357538,0.902597,0.603282
4,5,Cairo,0.639376,0.521473,0.758254,0.583601,0.576617,0.756938


In [35]:
import pandas as pd

def output_to_gpt(ranked_locations, phrases_df):
    loc = ranked_locations.iloc[:, 1].reset_index(drop=True).rename("locations")
    phr = phrases_df.iloc[:, 0].reset_index(drop=True).rename("keywords")
    n = min(len(loc), len(phr))
    return pd.DataFrame({loc.name: loc.iloc[:n], "keywords": phr.iloc[:n]})


In [36]:
output_to_gpt(ranked_locations, phrases_df)

,locations,keywords
0,Seoul,a lively city where someone could enjoy street...
1,Quebec City,a place with landmarks where someone could do ...
2,London,a scene of a river
3,Jeju Island,a scenic place where someone could do photography
4,Cairo,"a city, town or resort where someone could go ..."
